In [4]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

%matplotlib inline

In [5]:
FITS_DIR = "./downloaded-koi-lcss/"
fits_files = glob.glob(os.path.join(FITS_DIR, "*_llc.fits"))

In [6]:
for f in fits_files[:5]:
    print(f)

./downloaded-koi-lcss/kplr011904151-2010009091648_llc.fits
./downloaded-koi-lcss/kplr004139816-2011177032512_llc.fits
./downloaded-koi-lcss/kplr004139816-2012004120508_llc.fits
./downloaded-koi-lcss/kplr011904151-2011177032512_llc.fits
./downloaded-koi-lcss/kplr011904151-2012004120508_llc.fits


In [7]:
def inspect_koi_lightcurve(fits_filepath):
    """
    Opens a Kepler .fits file, extracts the time and flux arrays, 
    cleans out NaNs, and plots the light curve.
    """
    print(f"\n--- Inspecting: {os.path.basename(fits_filepath)} ---")
    
    try:
        with fits.open(fits_filepath) as hdul:
            # Print basic FITS info
            hdul.info()
            
            # Extract header metadata
            hdr = hdul[0].header
            kic_id = hdr.get('KEPLERID', 'Unknown')
            quarter = hdr.get('QUARTER', 'Unknown')
            
            # The lightcurve data is in the first extension (INDEX 1)
            data = hdul[1].data
            
            # Extract arrays and filter out NaNs (vital for Kepler data)
            mask = ~np.isnan(data['TIME']) & ~np.isnan(data['PDCSAP_FLUX'])
            
            time = data['TIME'][mask]
            flux = data['PDCSAP_FLUX'][mask]
            
            if 'PDCSAP_FLUX_ERR' in data.columns.names:
                flux_err = data['PDCSAP_FLUX_ERR'][mask]
            else:
                flux_err = None
                
            n_points = len(time)
            print(f"KIC: {kic_id} | Quarter: {quarter} | Valid Data Points: {n_points}")
            
            # Plotting the light curve
            plt.figure(figsize=(20, 5))
            plt.plot(time, flux, 'k.', markersize=2, label='PDCSAP Flux')
            
            if flux_err is not None:
                plt.fill_between(time, flux - flux_err, flux + flux_err, alpha=0.3, color='gray')
                
            plt.title(f"Light Curve for KIC {kic_id} (Quarter {quarter})")
            plt.xlabel("Time (BKJD)")
            plt.ylabel("Normalized Flux")
            plt.legend()
            plt.show()
            
            return {
                "KIC": kic_id,
                "Quarter": quarter,
                "time": time,
                "flux": flux,
                "flux_err": flux_err,
                "n_points": n_points
            }
            
    except Exception as e:
        print(f"Error reading file: {e}")
        return None

In [8]:
extracted_data = []

In [9]:
for filepath in fits_files[:10]: 
    try:
        with fits.open(filepath) as hdul:
            data = hdul[1].data
            mask = ~np.isnan(data['TIME']) & ~np.isnan(data['PDCSAP_FLUX'])
            
            extracted_data.append({
                "KIC": hdul[0].header.get('KEPLERID'),
                "quarter": hdul[0].header.get('QUARTER'),
                "time": data['TIME'][mask],
                "flux": data['PDCSAP_FLUX'][mask],
                "flux_err": data['PDCSAP_FLUX_ERR'][mask] if 'PDCSAP_FLUX_ERR' in data.columns.names else None,
                "n_points": len(data['TIME'][mask])
            })
    except Exception as e:
        pass # Skip corrupted files for now

In [10]:
df_kois = pd.DataFrame(extracted_data)
df_kois.head()

,KIC,quarter,time,flux,flux_err,n_points
0,11904151,4,"[352.3975382855133, 352.43840463818196, 352.45...","[575100.6, 575187.25, 575179.7, 575197.06, 575...","[20.944284, 20.981253, 20.946161, 20.930882, 2...",1008
1,4139816,9,"[808.5364370435782, 808.5568714091278, 808.577...","[5476.708, 5477.9116, 5479.4927, 5466.4385, 54...","[3.76462, 3.7641869, 3.7653987, 3.7662091, 3.7...",4608
2,4139816,11,"[1001.2287544391875, 1001.2491873596664, 1001....","[5780.9434, 5777.554, 5782.746, 5776.393, 5776...","[3.988103, 3.9886668, 3.9886744, 3.9874415, 3....",4473
3,11904151,9,"[808.5368768997796, 808.5573109466422, 808.577...","[548002.5, 548020.8, 548013.9, 548086.5, 54803...","[20.604708, 20.623955, 20.618187, 20.619051, 2...",4609
4,11904151,11,"[1001.2282819599204, 1001.2487152010071, 1001....","[522279.12, 522246.38, 522223.44, 522321.3, 52...","[19.909143, 19.911713, 19.909582, 19.945402, 1...",4473
